# Imports

In [ ]:
# ============================================================
# ANOM.0) Imports
# ============================================================

from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

# Project helpers (you already have these modules)
from src.bench.guardrails_artifacts import (
    build_guardrail_fn_registry,
    load_guardrail_spec,
)

# Optional (recommended): your parquet reader helper from the project
# If read_pq exists in your notebook utils, import it instead of redefining.
read_pq = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

print("Imports OK.")

# Paths + load V3 artifacts (spec + thresholds)

In [ ]:
# ============================================================
# ANOM.1) Load V3 artifacts (spec + thresholds)
# ============================================================

GUARDRAIL_SPEC_PATH = Path("artifacts/guardrails/v3/guardrail_v3_spec.json")
THRESHOLDS_PATH     = Path("artifacts/guardrails/v3/thresholds_v2.json")

assert GUARDRAIL_SPEC_PATH.exists(), f"Missing: {GUARDRAIL_SPEC_PATH}"
assert THRESHOLDS_PATH.exists(), f"Missing: {THRESHOLDS_PATH}"

registry = build_guardrail_fn_registry()

loaded = load_guardrail_spec(
    spec_path=GUARDRAIL_SPEC_PATH,
    thresholds_path=THRESHOLDS_PATH,
    fn_registry=registry,
)

guardrail_v3_rehydrated = loaded["guardrail"]
thresholds_v2_loaded = loaded["thresholds"]

EPS = float(thresholds_v2_loaded.get("epsilon", 1e-9))

print("Loaded guardrail:", guardrail_v3_rehydrated.get("name"), "| components:", len(guardrail_v3_rehydrated["components"]))
print("Threshold keys:", len(thresholds_v2_loaded))
print("EPS:", EPS)

# Load eval universe

In [ ]:
# ============================================================
# ANOM.2) Load anomaly universe (D/G + V3 scored frame)
# Preferred: load from a parquet you produced in the guardrails notebook
# ============================================================

# Option 1 (recommended): point to an explicit parquet file you saved.
# Update this path once, then keep the notebook stable.
EVAL_SCORED_DG_V3_PATH = Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")

if EVAL_SCORED_DG_V3_PATH.exists():
    eval_scored_DG_V3 = pd.read_parquet(EVAL_SCORED_DG_V3_PATH)
    print("Loaded:", EVAL_SCORED_DG_V3_PATH)
else:
    # Option 2: load from your failure-analysis parquet bundle if that's your current storage.
    # This assumes PARQUET_DIR points to the bundle you used in EVAL.1.
    # If you already have read_pq in this repo, we can use it; otherwise fallback to pd.read_parquet.
    PARQUET_DIR = Path("artifacts/failure_analysis_v2/parquet_rich_v2_20260309_093101")  # update if needed
    candidate = PARQUET_DIR / "failure_df_full_v2.parquet"
    assert candidate.exists(), f"Could not find eval universe parquet at {candidate} or {EVAL_SCORED_DG_V3_PATH}"

    eval_scored_DG_V3 = pd.read_parquet(candidate)
    print("Loaded:", candidate)
    print("NOTE: This is failure_df_full_v2. Confirm it reflects D/G+V3 expected_cost in this bundle.")

print("Rows:", len(eval_scored_DG_V3))
print("Cols:", eval_scored_DG_V3.shape[1])
display(eval_scored_DG_V3.head(3))

# Minimal schema sanity (so later cells fail fast)

In [ ]:
# ============================================================
# ANOM.3) Minimal schema sanity checks (fail fast)
# ============================================================

REQ = [
    "row_id", "Rndrng_NPI", "HCPCS_Cd", "Year",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    "services", "benes", "expected_cost_support_tier", "has_lag",
    "rbcs_family_desc", "state",
]

missing = [c for c in REQ if c not in eval_scored_DG_V3.columns]
assert not missing, f"eval_scored_DG_V3 is missing required columns: {missing}"

# Numeric coercions (defensive)
for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio","services","benes"]:
    eval_scored_DG_V3[c] = pd.to_numeric(eval_scored_DG_V3[c], errors="coerce")

eval_scored_DG_V3["has_lag"] = eval_scored_DG_V3["has_lag"].astype(bool)

# Basic finiteness checks
assert eval_scored_DG_V3["observed_cost"].notna().all(), "observed_cost has NaNs"
assert eval_scored_DG_V3["expected_cost"].notna().all(), "expected_cost has NaNs"
assert np.isfinite(eval_scored_DG_V3["oe_ratio"].to_numpy(dtype="float64")).all(), "oe_ratio has non-finite values"

print("Schema sanity OK.")

# ANOM.1 Compute confidence flags (new, auditable) + helper percentiles

In [ ]:
# ============================================================
# ANOM.1) Confidence flags + within-slice percentiles
# ============================================================

import numpy as np
import pandas as pd

df = eval_scored_DG_V3.copy()

# -----------------------------
# 1) Define a new "is_high_conf" flag (auditable, stable)
#    Tune thresholds later after looking at distributions.
# -----------------------------
MIN_SERVICES = 50
MIN_BENES = 20
HIGH_CONF_TIERS = {"high", "medium_high"}

df["is_high_conf"] = (
    df["expected_cost_support_tier"].astype(str).isin(HIGH_CONF_TIERS)
    & (df["services"].fillna(0) >= MIN_SERVICES)
    & (df["benes"].fillna(0) >= MIN_BENES)
)

# Optional: hot-start only (uncomment if you want this stricter definition)
# df["is_high_conf"] = df["is_high_conf"] & df["has_lag"].astype(bool)

# Keep your existing flag too
df["high_confidence_anomaly_candidate"] = df["high_confidence_anomaly_candidate"].astype(bool)

print("High-conf counts:")
display(pd.DataFrame({
    "flag": ["is_high_conf", "high_confidence_anomaly_candidate"],
    "n_true": [int(df["is_high_conf"].sum()), int(df["high_confidence_anomaly_candidate"].sum())],
    "pct_true_%": [float(df["is_high_conf"].mean()*100), float(df["high_confidence_anomaly_candidate"].mean()*100)]
}))

# -----------------------------
# 2) Create within-slice percentiles for oe_ratio and residual
#    Slices: (HCPCS_Cd, Year) as your most comparable unit.
#    If you prefer RBCS family, change slice_cols.
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    # percent rank in [0,1]; stable and easy
    return s.rank(pct=True, method="average")

df["oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["oe_ratio"].transform(_pct_rank)
df["resid_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["residual"].transform(_pct_rank)

# Convenience: top-x% flags
df["is_top_1pct_oe_in_slice"] = df["oe_pct_in_slice"] >= 0.99
df["is_top_1pct_resid_in_slice"] = df["resid_pct_in_slice"] >= 0.99

print("Percentile features created:", ["oe_pct_in_slice", "resid_pct_in_slice"])
display(df[["HCPCS_Cd","Year","oe_ratio","oe_pct_in_slice","residual","resid_pct_in_slice","services","benes","is_high_conf"]].head(5))

# Save back to the notebook namespace
eval_anom = df
print("Defined: eval_anom (copy of eval_scored_DG_V3 + anomaly features)")

# ANOM.2 Row-level “Top anomalies” action list

This is our “what should a stakeholder look at first” table

In [ ]:
# ============================================================
# ANOM.2) Row-level top anomalies (action list)
# ============================================================

df = eval_anom.copy()

# Scope: focus on over-expected (positive residual or high OE)
# You can tighten/loosen these later.
row_candidates = df[
    (df["oe_ratio"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
].copy()

# A simple, explainable scoring:
# - prioritize high OE percentile and residual percentile
# - boost if high-confidence
row_candidates["anom_score"] = (
    0.6 * row_candidates["oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

# Keep only over-expected direction (optional but usually desired)
row_candidates = row_candidates[(row_candidates["oe_ratio"] > 1.0) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows = (
    row_candidates.sort_values(["anom_score","oe_ratio","residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio",
        "oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
    ]]
)

print("Top anomalies (row-level):", len(top_anomalies_rows))
display(top_anomalies_rows.head(20))

# Save
anom_top_rows = top_anomalies_rows

In [ ]:
eval_scored_DG_V3.expected_cost.eq(0).sum()

In [ ]:
top_anomalies_rows.expected_cost.eq(0).sum()

In [ ]:
tolerance = 1e-3
(eval_scored_DG_V3["expected_cost"].abs() < tolerance).sum()

In [ ]:
tolerance = 1e-3
(top_anomalies_rows["expected_cost"].abs() < tolerance).sum()

In [ ]:
eval_scored_DG_V3.shape[0]

In [ ]:
top_anomalies_rows.shape[0]

In [ ]:
tolerances = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
counts_eval_scored_DG_V3 = []
counts_row_candidates = []

for tol in tolerances:
    counts_eval_scored_DG_V3.append((eval_scored_DG_V3["expected_cost"].abs() < tol).sum())
    counts_row_candidates.append((row_candidates["expected_cost"].abs() < tol).sum())

counts_eval_scored_DG_V3, counts_row_candidates



In [ ]:
tolerances = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
counts_eval_scored_DG_V3 = []
counts_row_candidates = []

for tol in tolerances:
    counts_eval_scored_DG_V3.append((eval_scored_DG_V3["observed_cost"].abs() < tol).sum())
    counts_row_candidates.append((row_candidates["observed_cost"].abs() < tol).sum())

counts_eval_scored_DG_V3, counts_row_candidates

# ANOM.3 Provider-level “who consistently pops” summary

This answers: “Which NPIs keep showing up across codes/years?”

In [ ]:
# ============================================================
# ANOM.3) Provider-level anomaly summary (who consistently pops)
# ============================================================

df = eval_anom.copy()

# Define "anomalous row" using your fast-track + percentile approach
df["is_row_anomalous"] = (
    (df["oe_ratio"] > 1.0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["oe_pct_in_slice"] >= 0.99)
)

provider_summary = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows=("is_row_anomalous","sum"),
        anom_rate_pct=("is_row_anomalous", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_residual=("residual","median"),
        total_services=("services","sum"),
        total_benes=("benes","sum"),
    )
    .reset_index()
)

# Rank: lots of anomalous rows + not just tiny footprint
provider_summary["provider_anom_score"] = (
    provider_summary["n_anom_rows"]
    + 0.25 * provider_summary["n_unique_codes"]
    + 0.25 * provider_summary["n_unique_years"]
)

TOP_N = 200
anom_top_providers = provider_summary.sort_values(
    ["provider_anom_score","n_anom_rows","anom_rate_pct","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers:", len(anom_top_providers))
display(anom_top_providers.head(30))

# Save
anom_top_providers = anom_top_providers

In [ ]:
anom_top_providers.columns